[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C60_Edge_Deployment_Consistency_Course/04_postprocess/04_postprocess_cpp.ipynb)

# 04 · 后处理对齐与 C++ 推理管线（NMS 变体 / letterbox 逆变换 / 端到端对拍）

目标：把「训练侧 Python 后处理」与「车端 C++ 后处理」之间**每一处会静默生效的分歧**
变成可以运行、可以断言、可以自动定位的东西。

本 notebook 你会亲手实现：
1. **两种 IoU 约定**（COCO 的 `offset=0` 与 VOC/Faster R-CNN 的 `offset=1`），
   量化差异并验证它按 $\mathcal{O}(1/s)$ 衰减 —— **只伤小目标**
2. **确定性 NMS**（排序键 = `(-score, index)`，消除并列歧义），
   并量化两种 IoU 约定给出的保留集差异
3. **top-k 位置实验**：截断放在 NMS 之前 vs 之后，密集帧上检出目标数的塌陷
4. **letterbox 正/逆变换**与**五个错误版本**，逐一算出偏移量
5. **class-wise / class-agnostic / 层次化 NMS**，含 `batched_nms` 的 offset trick
   与它的 **float32 数值悬崖**
6. **解码约定**：$\sigma(\sigma(x))$ 为什么「mAP 不变但线上炸」、坐标格式弄错的量化后果
7. **端到端对拍框架**：8 个阶段 + 逐阶段容差 + 自动定位第一个失配阶段，
   注入三种真实 bug 并验证每次都定位正确

> 心智模型：**后处理 bug 不会报错、不会崩溃、只会让你误判成模型问题。
> 防线不是「小心」，是「逐阶段对拍 + CI」。**

## 1 · IoU 的两种约定：`+1` 从哪来，差多少

`offset=1` 来自 Pascal VOC 的「像素索引闭区间」语义（框 `[0,0,7,7]` 覆盖 8 个像素）；
`offset=0` 是 COCO 的连续坐标半开区间语义。两者各自自洽，**混用就出事**。

In [ ]:
import numpy as np

def iou_pair(a, b, offset=0):
    # 两个 xyxy 框的 IoU。offset=0 → COCO 口径；offset=1 → VOC / Faster R-CNN 口径
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw = max(0.0, ix2 - ix1 + offset)
    ih = max(0.0, iy2 - iy1 + offset)
    inter = iw * ih
    aa = (a[2] - a[0] + offset) * (a[3] - a[1] + offset)
    ab = (b[2] - b[0] + offset) * (b[3] - b[1] + offset)
    return inter / (aa + ab - inter)

print('两个边长 s 的正方形框，沿对角线错开 d = s/4（相对位移固定 25%）\n')
print(f"{'边长 s':>8s} {'IoU(offset=0)':>15s} {'IoU(offset=1)':>15s} {'差值':>9s}")
for s in [8, 16, 32, 64]:
    d = s / 4
    A = [0, 0, s, s]
    B = [d, d, s + d, s + d]
    i0, i1 = iou_pair(A, B, 0), iou_pair(A, B, 1)
    print(f'{s:>8d} {i0:>15.4f} {i1:>15.4f} {i1 - i0:>9.4f}')

# —— 手算校验（分数都能口算）——
assert abs(iou_pair([0, 0, 8, 8], [2, 2, 10, 10], 0) - 36 / 92) < 1e-12    # 交 6x6, 并 64+64-36
assert abs(iou_pair([0, 0, 8, 8], [2, 2, 10, 10], 1) - 49 / 113) < 1e-12   # 交 7x7, 并 81+81-49
assert abs(iou_pair([0, 0, 64, 64], [16, 16, 80, 80], 0) - 36 / 92) < 1e-12
assert abs(iou_pair([0, 0, 64, 64], [16, 16, 80, 80], 1) - 2401 / 6049) < 1e-12
print('\n✅ 注意左列：offset=0 的 IoU 在所有尺度上**恒等于 0.3913** —— 它是尺度不变的。')
print('   offset=1 不是：那个 +1 是**绝对**像素量，对 8px 框是 12.5% 的膨胀，对 64px 框只有 1.6%。')

In [ ]:
# Δ(s) 到底怎么衰减：log-log 拟合
sizes = np.array([4, 8, 16, 32, 64, 128, 256], dtype=float)
deltas = []
for s in sizes:
    d = s / 4
    deltas.append(iou_pair([0, 0, s, s], [d, d, s + d, s + d], 1)
                  - iou_pair([0, 0, s, s], [d, d, s + d, s + d], 0))
deltas = np.array(deltas)

print(f"{'边长 s':>8s} {'Δ = IoU1 - IoU0':>17s} {'相对上一行':>11s}")
for i, s in enumerate(sizes):
    ratio = deltas[i - 1] / deltas[i] if i else float('nan')
    tail = f'{ratio:>11.2f}' if i else f"{'—':>11s}"
    print(f'{int(s):>8d} {deltas[i]:>17.5f}{tail}')

slope = float(np.polyfit(np.log(sizes), np.log(deltas), 1)[0])
print(f'\nlog-log 斜率 = {slope:.3f}   （-1 = 严格 1/s 衰减）')
assert -1.10 < slope < -0.85, slope
assert deltas[0] > 25 * deltas[-1], '4px 与 256px 的差异应相差一个数量级以上'
print('✅ Δ(s) ~ O(1/s)：**+1 约定是一个只伤害小目标的 bug**。')
print('   TSR 的标志常年 10-30 px —— 这正好是 Δ 最大的区间。')
print('   同类的还有 round/floor(±0.5px)、letterbox pad 奇偶(±0.5px)、align_corners(±0.5px)，会叠加。')

## 2 · 确定性 NMS，以及两种 IoU 约定造成的保留集差异

先把 NMS 写成一个**确定性函数**：排序键取 `(-score, index)`，这样并列分数不再有歧义。
（模块正文第 3 节：INT8 输出只有约 256 个离散值，并列是常态，
不做这一步的话 Python 与 C++ 的输出就不可复现。）

In [ ]:
def iou_1_to_n(box, boxes, offset=0):
    # box: (4,) xyxy;  boxes: (N,4)  ->  (N,) IoU
    x1 = np.maximum(box[0], boxes[:, 0]); y1 = np.maximum(box[1], boxes[:, 1])
    x2 = np.minimum(box[2], boxes[:, 2]); y2 = np.minimum(box[3], boxes[:, 3])
    w = np.clip(x2 - x1 + offset, 0, None); h = np.clip(y2 - y1 + offset, 0, None)
    inter = w * h
    a0 = (box[2] - box[0] + offset) * (box[3] - box[1] + offset)
    a1 = (boxes[:, 2] - boxes[:, 0] + offset) * (boxes[:, 3] - boxes[:, 1] + offset)
    return inter / np.maximum(a0 + a1 - inter, 1e-12)

def score_order(scores):
    # **确定性全序**：先按分数降序，分数相同时按原始下标升序。
    # np.lexsort 的最后一个键是主键。
    return np.lexsort((np.arange(len(scores)), -np.asarray(scores, dtype=float)))

def nms(boxes, scores, iou_thr=0.5, offset=0):
    order = score_order(scores)
    keep = []
    while order.size:
        i = order[0]; keep.append(int(i))
        rest = order[1:]
        if rest.size == 0:
            break
        ious = iou_1_to_n(boxes[i], boxes[rest], offset)
        order = rest[ious <= iou_thr]          # 抑制条件是 > thr（保留 <= thr）
    return np.array(keep, dtype=int)

# —— 手算校验 ——
B = np.array([[0, 0, 10, 10], [1, 1, 11, 11], [50, 50, 60, 60]], dtype=float)
S = np.array([0.9, 0.8, 0.7])
assert nms(B, S, 0.5).tolist() == [0, 2]
assert abs(iou_1_to_n(B[0], B[1:2], 0)[0] - 81 / 119) < 1e-12
# 并列分数：确定性排序保证结果与输入顺序无关地可复现
tie_b = np.array([[0, 0, 8, 8], [1, 1, 9, 9], [40, 40, 48, 48]], dtype=float)
tie_s = np.array([0.5, 0.5, 0.5])
assert nms(tie_b, tie_s, 0.5).tolist() == [0, 2], '并列时应按 index 决胜，保留 0 号'
print('✅ 确定性 NMS 就位：排序键 = (-score, index)，并列不再有歧义。')

In [ ]:
# —— 决定性演示：同一对框，阈值卡在两种约定之间 ——
A, Bx = [0, 0, 8, 8], [2, 2, 10, 10]
THR = 0.41                                   # 0.3913 < 0.41 < 0.4336
bb = np.array([A, Bx], dtype=float)
ss = np.array([0.9, 0.8])
k0 = nms(bb, ss, THR, offset=0)
k1 = nms(bb, ss, THR, offset=1)
print(f'iou_thr={THR}:  IoU0={iou_pair(A,Bx,0):.4f}  IoU1={iou_pair(A,Bx,1):.4f}')
print(f'  offset=0 -> keep {k0.tolist()}   (0.3913 <= 0.41，**不抑制**，输出 2 个框)')
print(f'  offset=1 -> keep {k1.tolist()}      (0.4336 >  0.41，**抑制**，输出 1 个框)')
assert k0.tolist() == [0, 1] and k1.tolist() == [0]
print('\n⚠️  同一份权重、同一张图、同一个阈值 —— 两侧输出的框数不同。没有任何报错。')

In [ ]:
def tsr_scene(n_sign, rng, props=14, px_lo=8, px_hi=28, W=1920, H=1080):
    # 合成一帧 TSR 检测器的原始候选：
    #   每块牌子周围 props 个抖动候选（抖动幅度 ∝ 框尺寸），外加随场景变多的背景误检
    ctr = rng.uniform([80, 80], [W - 80, H - 80], size=(n_sign, 2))
    sz = rng.uniform(px_lo, px_hi, size=(n_sign, 1))
    c = np.repeat(ctr, props, axis=0); s = np.repeat(sz, props, axis=0)
    c = c + rng.normal(0, 0.10, size=c.shape) * s
    s = s * np.exp(rng.normal(0, 0.08, size=s.shape))
    fb = np.concatenate([c - s / 2, c + s / 2], axis=1)
    peak = rng.beta(2.5, 1.5, size=(n_sign, 1))
    fs = (peak * rng.uniform(0.30, 1.0, size=(n_sign, props))).ravel()
    gid = np.repeat(np.arange(n_sign), props)
    n_bg = 20 + 4 * n_sign
    bc = rng.uniform([0, 0], [W, H], size=(n_bg, 2)); bs = rng.uniform(8, 30, size=(n_bg, 1))
    bb = np.concatenate([bc - bs / 2, bc + bs / 2], axis=1)
    return (np.vstack([fb, bb]),
            np.concatenate([fs, rng.uniform(0.02, 0.30, n_bg)]),
            np.concatenate([gid, np.full(n_bg, -1)]))

# 20 个种子上统计两种约定的保留集差异
tot_sym, tot_keep = 0, 0
for seed in range(20):
    b, s, g = tsr_scene(40, np.random.default_rng(seed))
    m = s >= 0.25
    k0 = set(nms(b[m], s[m], 0.42, 0).tolist())
    k1 = set(nms(b[m], s[m], 0.42, 1).tolist())
    tot_sym += len(k0 ^ k1); tot_keep += len(k0)
print(f'20 帧合计：offset=0 保留 {tot_keep} 个框，两种约定的对称差 {tot_sym} 个')
print(f'-> 约 {tot_sym / tot_keep:.1%} 的保留框会因为一个 +1 而不同')
assert tot_sym > 0, '小框密集场景下两种约定必然给出不同的保留集'
print('✅ 这就是「小目标桶掉点、大目标桶不掉」的一种真实成因。')

## 3 · top-k 的位置：截断放在 NMS 之前会吃掉密集场景

同一个 `top-k`，放在 NMS **之前**是「从未去重的候选里取 k 个」，
放在 **之后**是「从已去重的目标里取 k 个」。前者在密集帧上会静默漏掉大批目标。

In [ ]:
def n_covered(keep_idx, gid):
    # 保留集覆盖了多少个不同的真实目标
    g = gid[keep_idx]
    return len(set(g[g >= 0].tolist()))

def pipeline(boxes, scores, gid, score_thr=0.25, topk_before=None,
             iou_thr=0.5, max_det=300, offset=0):
    m = scores >= score_thr
    b, s, g = boxes[m], scores[m], gid[m]
    n_pass = len(s)
    if topk_before is not None:                        # 方案 B：NMS **之前**截断
        sel = score_order(s)[:topk_before]
        b, s, g = b[sel], s[sel], g[sel]
    keep = nms(b, s, iou_thr, offset)[:max_det]        # 方案 A：只在 NMS **之后**截断
    return n_pass, len(keep), n_covered(keep, g)

DENSE = tsr_scene(60, np.random.default_rng(7), props=20)     # 城市路口：60 块牌子
SPARSE = tsr_scene(4, np.random.default_rng(7), props=20)     # 高速：4 块牌子

for name, scene, n_gt in [('密集帧（60 块牌子）', DENSE, 60), ('稀疏帧（4 块牌子）', SPARSE, 4)]:
    print(f'\n{name}')
    print(f"{'NMS 前 topK':>13s} {'过阈 N':>8s} {'输出框数':>9s} {'覆盖目标数':>11s} {'召回':>8s}")
    prev_cov = -1
    for tk in [100, 200, 300, 600, 1200, None]:
        n_pass, n_out, cov = pipeline(*scene, topk_before=tk)
        label = 'None(不截断)' if tk is None else str(tk)
        print(f'{label:>13s} {n_pass:>8d} {n_out:>9d} {cov:>11d} {cov / n_gt:>7.1%}')
        assert cov >= prev_cov, 'topK 越大，覆盖的目标数必须单调不减'
        prev_cov = cov

n_pass_d, _, cov_d100 = pipeline(*DENSE, topk_before=100)
_, _, cov_dinf = pipeline(*DENSE, topk_before=None)
n_pass_s, _, cov_s100 = pipeline(*SPARSE, topk_before=100)
_, _, cov_sinf = pipeline(*SPARSE, topk_before=None)
assert cov_d100 < 0.65 * cov_dinf, (cov_d100, cov_dinf)
assert cov_s100 == cov_sinf, '稀疏帧上 topK=100 根本不生效 -> 两方案完全一致'
print(f'\n⚠️  密集帧：topK=100 时覆盖 {cov_d100}/60，不截断时 {cov_dinf}/60 —— **静默丢了一大半**')
print(f'✅ 稀疏帧：topK=100 时覆盖 {cov_s100}/4，不截断时 {cov_sinf}/4 —— **完全一样**')
print('   -> 这个 bug **只在密集场景发作**，而离线评测集里这种帧本来就少 -> mAP 上看不出来。')
print('   记住：NMS 前的 topK 按「最大过阈候选数」留 2-3 倍余量（几千）；')
print('         NMS 后的 max_det 才按「单帧最大目标数」设（几百）。两者不能互相替代。')

In [ ]:
# per-class 还是 global：TSR 的 200 类会怎么被挤
rng = np.random.default_rng(3)
N_CLS = 200
b, s, g = tsr_scene(50, np.random.default_rng(11), props=20)
# 让类别高度不均衡：多数候选集中在 3 个常见类（限速 60/80/100），其余是长尾
cls = np.where(rng.random(len(s)) < 0.75,
               rng.integers(0, 3, len(s)),
               rng.integers(3, N_CLS, len(s)))
m = s >= 0.25
b, s, g, cls = b[m], s[m], g[m], cls[m]

def topk_global(s, k):
    return score_order(s)[:k]

def topk_per_class(s, cls, k_per):
    out = []
    for c in np.unique(cls):
        idx = np.where(cls == c)[0]
        out.append(idx[score_order(s[idx])[:k_per]])
    return np.concatenate(out)

gsel = topk_global(s, 300)
psel = topk_per_class(s, cls, 5)
print(f'过阈候选 {len(s)} 个，覆盖 {len(np.unique(cls))} 个类别')
print(f'global top-300      : 选中 {len(gsel):4d} 个，覆盖 {len(np.unique(cls[gsel])):3d} 个类别')
print(f'per-class top-5     : 选中 {len(psel):4d} 个，覆盖 {len(np.unique(cls[psel])):3d} 个类别')
assert len(np.unique(cls[psel])) > len(np.unique(cls[gsel])), 'per-class 应覆盖更多类别'
assert len(psel) != len(gsel)
print('\n⚠️  global top-k 会被头部类别吃掉名额 —— TSR 的 200 类里，长尾类先出局。')
print('   mmdetection 的做法是第三种切法：**按 FPN 层级**各取 top-k，')
print('   保证小目标层（stride 8）不会被大目标层挤掉 —— 对 TSR 比按类别切更有价值。')

## 4 · letterbox 正/逆变换：五个错误版本各偏多少

$1920\times1080 \to 640\times640$：$r = 1/3$，$p_x = 0$，$p_y = 140$。
测试框在网络坐标下是 `[100, 200, 140, 240]`，正确还原是 `[300, 180, 420, 300]`。

In [ ]:
def letterbox_params(src_hw, dst_hw):
    (H, W), (Hn, Wn) = src_hw, dst_hw
    r = min(Wn / W, Hn / H)
    new_w, new_h = round(W * r), round(H * r)
    return dict(r=r, pad_x=(Wn - new_w) / 2.0, pad_y=(Hn - new_h) / 2.0,
                new_w=new_w, new_h=new_h, W=W, H=H, Wn=Wn, Hn=Hn)

def lb_forward(box, p):
    # 原图 xyxy -> 网络输入 xyxy
    r, px, py = p['r'], p['pad_x'], p['pad_y']
    return np.array([box[0] * r + px, box[1] * r + py, box[2] * r + px, box[3] * r + py])

def lb_inverse(box, p):
    # 网络输入 xyxy -> 原图 xyxy   ** 先减 pad，再除 r **
    r, px, py = p['r'], p['pad_x'], p['pad_y']
    return np.array([(box[0] - px) / r, (box[1] - py) / r,
                     (box[2] - px) / r, (box[3] - py) / r])

P640 = letterbox_params((1080, 1920), (640, 640))
print('letterbox 参数:', {k: round(v, 4) if isinstance(v, float) else v for k, v in P640.items()})
assert abs(P640['r'] - 1 / 3) < 1e-12
assert P640['new_w'] == 640 and P640['new_h'] == 360
assert P640['pad_x'] == 0.0 and P640['pad_y'] == 140.0

gt = np.array([300.0, 180.0, 420.0, 300.0])
net = lb_forward(gt, P640)
print('\n原图框', gt, ' -> 网络坐标', net)
assert np.allclose(net, [100, 200, 140, 240])
assert np.allclose(lb_inverse(net, P640), gt)

# —— 往返不变量测试：20 行代码挡住 E1~E4 全部四种错法 ——
rng = np.random.default_rng(0)
for _ in range(2000):
    x1, y1 = rng.uniform(0, 1800), rng.uniform(0, 1000)
    box = np.array([x1, y1, x1 + rng.uniform(4, 100), y1 + rng.uniform(4, 100)])
    assert np.allclose(lb_inverse(lb_forward(box, P640), P640), box, atol=1e-9)
print('✅ 往返一致性 inverse(forward(b)) == b 在 2000 个随机框上成立（atol=1e-9）')

In [ ]:
# —— 五个错误版本 ——
def inv_E1(box, p):    # 漏减 pad
    return box / p['r']

def inv_E2(box, p):    # 顺序反：先除 r 再减 pad
    r, px, py = p['r'], p['pad_x'], p['pad_y']
    return np.array([box[0] / r - px, box[1] / r - py, box[2] / r - px, box[3] / r - py])

def inv_E3(box, p):    # pad 减了两遍（把「单边 pad」当成「总 pad」）
    r, px, py = p['r'], p['pad_x'], p['pad_y']
    return np.array([(box[0] - 2 * px) / r, (box[1] - 2 * py) / r,
                     (box[2] - 2 * px) / r, (box[3] - 2 * py) / r])

def inv_E4(box, p):    # 用了 stretch（非等比）缩放比
    sx, sy = p['Wn'] / p['W'], p['Hn'] / p['H']
    return np.array([box[0] / sx, box[1] / sy, box[2] / sx, box[3] / sy])

print(f"{'版本':<26s} {'还原结果':<34s} {'Δy (中心)':>11s} {'与真值 IoU':>11s}")
def iou_np(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    it = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - it
    return it / ua if ua > 0 else 0.0

for name, fn in [('✅ 正确', lb_inverse), ('E1 漏减 pad', inv_E1), ('E2 顺序反', inv_E2),
                 ('E3 pad 减两遍', inv_E3), ('E4 stretch 比例', inv_E4)]:
    out = fn(net, P640)
    dy = (out[1] + out[3]) / 2 - (gt[1] + gt[3]) / 2
    print(f'{name:<26s} {str(np.round(out, 1)):<34s} {dy:>11.1f} {iou_np(out, gt):>11.3f}')

r, py = P640['r'], P640['pad_y']
assert np.allclose(inv_E1(net, P640), [300, 600, 420, 720])
assert np.allclose(inv_E2(net, P640), [300, 460, 420, 580])
assert np.allclose(inv_E3(net, P640), [300, -240, 420, -120])
assert np.allclose(inv_E4(net, P640), [300, 337.5, 420, 405])
assert abs(py / r - 420.0) < 1e-9                     # E1 的偏移量 = p_y / r
assert abs(py * (1 - r) / r - 280.0) < 1e-9           # E2 的偏移量 = p_y(1-r)/r
print('\n✅ 每个错误版本的偏移量都有闭式解：E1=+p/r=420, E2=+p(1-r)/r=280, E3=-p/r=-420。')
print('⚠️  E1/E3 偏得太狠一眼可见；**真正能在量产里活很久的是 E4 和亚像素级的取整分歧**。')

In [ ]:
# —— E5：clip 的位置 ——
def clip_wrong(box, p):        # 先在网络坐标下 clip 到 [0, 640]，再逆变换，然后当成有效检测输出
    b = np.clip(box, 0, [p['Wn'], p['Hn'], p['Wn'], p['Hn']])
    b = lb_inverse(b, p)
    return np.clip(b, 0, [p['W'], p['H'], p['W'], p['H']])     # 最终仍要落回图像范围

def clip_right(box, p):        # 先逆变换，再 clip 到原图，最后**丢弃退化框**
    b = lb_inverse(box, p)
    b = np.clip(b, 0, [p['W'], p['H'], p['W'], p['H']])
    return b if (b[2] - b[0] > 1 and b[3] - b[1] > 1) else None

edge = np.array([300.0, 20.0, 340.0, 100.0])          # 完全落在上方 padding 区（y < pad_y = 140）
wrong = clip_wrong(edge, P640)
print('一个完全落在 padding 区里的候选框（网络坐标）:', edge)
print('  错误顺序（网络坐标下 clip）:', np.round(wrong, 1),
      f' -> 高度 {wrong[3]-wrong[1]:.1f} px，**作为一个贴在图像上边缘的假检测被上报**')
print('  正确顺序（原图坐标下 clip + 退化过滤）:', clip_right(edge, P640), ' -> 直接丢弃')
assert clip_right(edge, P640) is None
assert wrong[3] - wrong[1] < 1e-9, '错误顺序会产出一个零高度的退化框，而它仍然进入了输出'
print('\n✅ clip 的语义是「限制在**真实图像**范围内」。')
print('   在网络坐标下 clip = 允许框延伸到 padding 区 —— 而 padding 区在原图上根本不存在。')

In [ ]:
# —— 亚像素偏移的代价：为什么 4px 在 TSR 里是灾难 ——
def iou_shift(s, d):
    # 边长 s 的框沿对角线整体偏移 d 像素后，与真值的 IoU
    inter = max(0.0, s - d) ** 2
    return inter / (2 * s * s - inter)

print(f"{'框边长':>8s} {'偏 1px':>9s} {'偏 2px':>9s} {'偏 4px':>9s} {'IoU=0.5 口径下偏 4px 算命中?':>28s}")
for s in [12, 20, 32, 64, 100]:
    hits = 'YES' if iou_shift(s, 4) >= 0.5 else '**NO -> 算完全漏检**'
    print(f'{s:>8d} {iou_shift(s,1):>9.3f} {iou_shift(s,2):>9.3f} {iou_shift(s,4):>9.3f} {hits:>28s}')

assert abs(iou_shift(12, 4) - 64 / 224) < 1e-12
assert abs(iou_shift(100, 4) - 9216 / 10784) < 1e-12
assert iou_shift(12, 4) < 0.5 and iou_shift(100, 4) > 0.8
print(f'\n⚠️  同一个 4 px 的系统性偏移：')
print(f'   100 px 的近处大牌 -> IoU {iou_shift(100,4):.3f}，任何口径下都算命中，**你永远不会发现**')
print(f'    12 px 的远处小牌 -> IoU {iou_shift(12,4):.3f}，IoU=0.5 口径下**直接算完全漏检**')
print('✅ 报告上会写「小目标 AP 掉一半，大目标不变」—— 和「模型对小目标不行」形态完全一样。')
print('   结论：凡是「只有小目标桶掉点」，第一怀疑对象永远是坐标变换，不是模型能力。')

## 5 · class-wise / class-agnostic / 层次化 NMS

`batched_nms` 的 offset trick：给每个框加上 `class_id × Δ`，把不同类别推到互不相交的坐标区间，
再做**一次** class-agnostic NMS —— 与逐类循环严格等价，但只有一次 kernel launch。
**前提是 Δ 选得对：选大了会掉进 float32 的数值悬崖。**

In [ ]:
def nms_classwise(boxes, scores, cls, iou_thr=0.5, offset=0):
    # 朴素版：逐类循环
    keep = []
    for c in np.unique(cls):
        idx = np.where(cls == c)[0]
        keep.append(idx[nms(boxes[idx], scores[idx], iou_thr, offset)])
    return np.sort(np.concatenate(keep))

def nms_batched(boxes, scores, cls, iou_thr=0.5, offset=0, delta=None):
    # offset trick：一次调用做完 class-wise
    if delta is None:
        delta = float(boxes.max()) + 1.0          # **Δ = 图像最大坐标 + 1，不是一个拍脑袋的大数**
    shifted = boxes + (cls.astype(np.float64) * delta)[:, None]
    return np.sort(nms(shifted, scores, iou_thr, offset))

def nms_agnostic(boxes, scores, cls, iou_thr=0.5, offset=0):
    return np.sort(nms(boxes, scores, iou_thr, offset))

rng = np.random.default_rng(21)
b, s, g = tsr_scene(30, np.random.default_rng(5), props=12)
cls = rng.integers(0, 8, len(s))
m = s >= 0.25
b, s, cls = b[m], s[m], cls[m]

kc = nms_classwise(b, s, cls, 0.5)
kb = nms_batched(b, s, cls, 0.5)
ka = nms_agnostic(b, s, cls, 0.5)
print(f'候选 {len(s)} 个 / {len(np.unique(cls))} 类')
print(f'  class-wise（逐类循环）: {len(kc)} 个框，调用 NMS {len(np.unique(cls))} 次')
print(f'  batched（offset trick）: {len(kb)} 个框，调用 NMS 1 次')
print(f'  class-agnostic         : {len(ka)} 个框')
assert np.array_equal(kc, kb), 'offset trick 必须与逐类循环**严格等价**'
assert len(ka) < len(kc), 'agnostic 跨类抑制，保留框必然更少'
print('\n✅ offset trick 与逐类循环逐元素相同（不是「差不多」，是严格相等）。')

In [ ]:
# —— float32 的数值悬崖：Δ 选大了会静默摧毁 IoU ——
box32 = np.array([100.0, 100.0, 108.0, 108.0], dtype=np.float32)     # 一个 8x8 的小标志框
print(f"{'Δ':>12s} {'类别 ID':>8s} {'偏移后坐标量级':>15s} {'ULP(px)':>9s} {'偏移后框宽':>11s}")
for delta, cid in [(1921.0, 0), (1921.0, 199), (1e6, 199), (1e6, 1000)]:
    off = np.float32(delta * cid)
    shifted = (box32 + off).astype(np.float32)
    w = float(shifted[2] - shifted[0])
    mag = float(shifted[0])
    ulp = float(np.spacing(np.float32(max(mag, 1.0))))
    print(f'{delta:>12.0f} {cid:>8d} {mag:>15.3e} {ulp:>9.3f} {w:>11.3f}')

bad = (box32 + np.float32(1e6 * 1000)).astype(np.float32)
assert float(bad[2] - bad[0]) == 0.0, 'Δ=1e6、类别 1000 时，8px 的框宽度被舍入成 0'
good = (box32 + np.float32(1921.0 * 199)).astype(np.float32)
assert abs(float(good[2] - good[0]) - 8.0) < 0.01
print('\n⚠️  Δ=1e6 且类别 ID=1000 时坐标到 1e9，float32 的 ULP ≈ 64 px：')
print('    x1 和 x2 被舍入到**同一个浮点数** -> 框宽 0 -> 面积 0 -> IoU 恒为 0 -> **NMS 完全失效**')
print('    而且不报任何错，输出的框数会莫名其妙地暴涨。')
print('✅ 正确做法：Δ = 图像最大边 + 1；或者把坐标量化到 int32 再加偏移。')

In [ ]:
# —— TSR 的层次化 NMS：超类内 agnostic，跨超类 class-wise ——
# 场景：一块「限速 60」牌同时被检成 限速60(0.52) / 限速80(0.47)；
#       它下方紧贴一块「货车」辅助牌（与主牌 IoU 约 0.45）
CLS_NAME = {0: '限速60', 1: '限速80', 2: '禁止左转', 3: '辅助牌·货车'}
SUPER = {0: 'speed', 1: 'speed', 2: 'prohibit', 3: 'plate'}     # 超类划分表

boxes_t = np.array([[100, 100, 118, 118],      # 限速60
                    [101,  99, 119, 117],      # 限速80（同一块物理牌子）
                    [100, 116, 118, 128],      # 辅助牌（在主牌下方，IoU 与主牌 ~0.14）
                    [400, 200, 424, 224]],     # 另一处的禁止左转
                   dtype=float)
scores_t = np.array([0.52, 0.47, 0.61, 0.80])
cls_t = np.array([0, 1, 3, 2])

def nms_hierarchical(boxes, scores, cls, super_map, iou_thr=0.5, offset=0):
    sup = np.array([super_map[int(c)] for c in cls])
    keep = []
    for u in np.unique(sup):                        # 超类之间互不抑制
        idx = np.where(sup == u)[0]
        keep.append(idx[nms(boxes[idx], scores[idx], iou_thr, offset)])   # 超类内 agnostic
    return np.sort(np.concatenate(keep))

for name, k in [('class-wise ', nms_classwise(boxes_t, scores_t, cls_t, 0.5)),
                ('agnostic   ', nms_agnostic(boxes_t, scores_t, cls_t, 0.5)),
                ('层次化      ', nms_hierarchical(boxes_t, scores_t, cls_t, SUPER, 0.5))]:
    names = [CLS_NAME[int(cls_t[i])] for i in k]
    print(f'{name}: {len(k)} 个框  ->  {names}')

kw = nms_classwise(boxes_t, scores_t, cls_t, 0.5)
ka2 = nms_agnostic(boxes_t, scores_t, cls_t, 0.5)
kh = nms_hierarchical(boxes_t, scores_t, cls_t, SUPER, 0.5)
assert set(kw.tolist()) == {0, 1, 2, 3}, 'class-wise 什么都不抑制：把消歧责任甩给下游'
assert 1 not in kh, '层次化：限速80 被同超类的限速60 抑制'
assert 2 in kh and 3 in kh, '层次化：辅助牌与禁止左转跨超类，必须保留'
assert len(kh) == 3
print('\n⚠️  class-wise：下游拿到「这里既是限速60又是限速80」，而下游没有消歧所需的信息。')
print('⚠️  agnostic ：本例阈值下辅助牌侥幸活着，但只要主辅牌贴得更近就会被整块抹掉 ——')
print('    而辅助牌承载的是限定条件（「7:00-20:00」「货车」），删掉它，主牌语义就是错的。')
print('✅ 层次化是唯一同时解决两个问题的模式。')
print('   工程补充：**被抑制的类别与分数不该丢**，要作为「竞争假设」附在保留框上传给下游 ——')
print('   单帧上 0.52 vs 0.47 几乎是抛硬币，连续 10 帧的贝叶斯累积才能把它分开（C55 模块 04）。')

## 6 · 解码约定：$\sigma(\sigma(x))$ 与坐标格式

本节两个实验，都是「不报错、不崩溃、指标还挺好看」的那一类 bug。

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

rng = np.random.default_rng(2024)
logits = rng.normal(-2.0, 2.0, size=8400)               # 一个典型的检测头 logit 分布
single = sigmoid(logits)                                # 正确：只做一次
double = sigmoid(single)                                # bug：模型里已经有 Sigmoid，外面又做了一次

print(f"{'':<18s} {'min':>8s} {'max':>8s} {'>=0.30 的比例':>14s}")
print(f"{'σ(x)   正确':<18s} {single.min():>8.4f} {single.max():>8.4f} {(single>=0.3).mean():>13.1%}")
print(f"{'σ(σ(x)) 出事了':<18s} {double.min():>8.4f} {double.max():>8.4f} {(double>=0.3).mean():>13.1%}")

assert double.min() > 0.5 and double.max() < 0.7311
assert (double >= 0.3).mean() == 1.0, '所有候选全部过阈 -> 一个框都筛不掉'
assert (single >= 0.3).mean() < 0.40

# **排序完全不变** —— 所以基于排序的指标（AP）几乎不动
o1 = np.argsort(-single, kind='stable'); o2 = np.argsort(-double, kind='stable')
assert np.array_equal(o1, o2), 'σ 是严格单调的，排序必然不变'

labels = (rng.random(8400) < sigmoid(logits * 0.9)).astype(int)      # 与 logit 相关的伪标签
def average_precision(order, y):
    y = y[order]; tp = np.cumsum(y); prec = tp / np.arange(1, len(y) + 1)
    return float((prec * y).sum() / max(y.sum(), 1))
ap1, ap2 = average_precision(o1, labels), average_precision(o2, labels)
print(f'\nAP(正确) = {ap1:.6f}    AP(σ∘σ) = {ap2:.6f}    差 = {abs(ap1-ap2):.2e}')
assert abs(ap1 - ap2) < 1e-12, 'AP 是纯排序指标，对单调变换免疫'
print('\n⚠️  **离线评测报告是绿的，线上却炸了**：')
print('    候选数从几百涨到 8400 -> NMS 是 O(NK) -> 延迟 p99 爆表；同时背景框全部过阈 -> FP 暴涨。')
print('✅ 诊断只要一行：`print(scores.min())`。最小值 > 0.5 就抓到它了。')
print('   反向错误同样常见：模型输出 logits 却拿去和 0.3 比 —— 等于把阈值悄悄抬到 σ(0.3)=0.574。')

In [ ]:
# —— 坐标格式：四种，没有一种自带标签 ——
def cxcywh_to_xyxy(b):
    cx, cy, w, h = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    return np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1)

def xywh_to_xyxy(b):
    x, y, w, h = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    return np.stack([x, y, x + w, y + h], axis=1)

rng = np.random.default_rng(9)
n = 500
cx = rng.uniform(200, 1700, n); cy = rng.uniform(150, 950, n)
w = rng.uniform(10, 40, n); h = w * rng.uniform(0.9, 1.1, n)
gt_xyxy = cxcywh_to_xyxy(np.stack([cx, cy, w, h], axis=1))
gt_cxcywh = np.stack([cx, cy, w, h], axis=1)
gt_xywh = np.stack([gt_xyxy[:, 0], gt_xyxy[:, 1], w, h], axis=1)

def mean_iou(a, b):
    return float(np.mean([iou_np(a[i], b[i]) for i in range(len(a))]))

def frac_degenerate(b):
    return float(np.mean((b[:, 2] <= b[:, 0]) | (b[:, 3] <= b[:, 1])))

print(f"{'把它当成 xyxy 用':<24s} {'与真值平均 IoU':>15s} {'退化框(负/零面积)占比':>22s}")
print(f"{'✅ 真的是 xyxy':<24s} {mean_iou(gt_xyxy, gt_xyxy):>15.4f} {frac_degenerate(gt_xyxy):>21.1%}")
print(f"{'cxcywh 当 xyxy':<24s} {mean_iou(gt_cxcywh, gt_xyxy):>15.4f} {frac_degenerate(gt_cxcywh):>21.1%}")
print(f"{'xywh   当 xyxy':<24s} {mean_iou(gt_xywh, gt_xyxy):>15.4f} {frac_degenerate(gt_xywh):>21.1%}")
print(f"{'yxyx   当 xyxy':<24s} {mean_iou(gt_xyxy[:, [1,0,3,2]], gt_xyxy):>15.4f} "
      f"{frac_degenerate(gt_xyxy[:, [1,0,3,2]]):>21.1%}")

assert mean_iou(gt_cxcywh, gt_xyxy) < 0.05, 'cxcywh 当 xyxy 用，小目标 IoU 直接归零'
assert frac_degenerate(gt_xywh) > 0.95, 'xywh 当 xyxy 用：w << x1，几乎全是负面积框'
assert mean_iou(gt_xyxy[:, [1, 0, 3, 2]], gt_xyxy) < 0.05
print('\n⚠️  `xywh / cxcywh 当 xyxy` 的症状特别值得记：像素坐标下 w << x1，')
print('    框全是负面积 -> IoU 恒为 0 -> **NMS 一个都不抑制** -> 满屏重复框。')
print('    看起来像「NMS 坏了」，其实是坐标格式。（归一化坐标下则是框整体挤向左上角。）')
print('    `yxyx 当 xyxy` 不产生退化框，只是沿主对角线镜像 —— 更隐蔽，近方形图上尤其难看出来。')
print('✅ 最佳工程答案是 Detectron2 的做法：把坐标格式做成**显式类型**（BoxMode），')
print('   而不是靠注释和口头约定。类型系统能挡住的 bug，就别靠人挡。')

## 7 · 端到端对拍框架：8 个阶段，自动定位第一个失配

下面搭一条**完整的迷你检测管线**（合成图 → letterbox → 归一化 → 迷你「模型」→ 解码 → NMS → 逆变换），
然后往里注入四种真实 bug，验证对拍器每次都能定位到**正确的阶段**。

管线参数：原图 $320\times180$ → 网络 $128\times128$，于是 $r=0.4$、$p_x=0$、$p_y=28$。

In [ ]:
# ── 合成图像：暗背景上贴几块「标志」（BGR uint8）──
#    紫色块 = 同时触发两个类别头（正是「限速60 / 限速80」那种细类互斥的情形）
PLANTED = [(60,  60, 30, 'red'), (200,  50, 26, 'blue'),
           (120, 120, 30, 'purple'), (250, 130, 22, 'red')]
BGR = {'red': (40, 40, 220), 'blue': (220, 60, 40), 'purple': (200, 40, 200)}

def make_image(seed=0, W=320, H=180):
    r = np.random.default_rng(seed)
    img = r.integers(30, 70, size=(H, W, 3)).astype(np.uint8)
    for cx, cy, s, col in PLANTED:
        img[cy - s // 2:cy + s // 2, cx - s // 2:cx + s // 2, :] = np.array(BGR[col], np.uint8)
    return img

def lb_image(img, p):
    H, W = img.shape[:2]
    out = np.full((p['Hn'], p['Wn'], 3), 114, np.uint8)
    ys = np.clip((np.arange(p['new_h']) / p['r']).astype(int), 0, H - 1)
    xs = np.clip((np.arange(p['new_w']) / p['r']).astype(int), 0, W - 1)
    y0, x0 = int(p['pad_y']), int(p['pad_x'])
    out[y0:y0 + p['new_h'], x0:x0 + p['new_w']] = img[np.ix_(ys, xs)]
    return out

def patch_mean(ch, stride=4, k=8):
    # ch: (Hn, Wn) -> (Hn/stride, Wn/stride) 的滑窗均值（k x k 窗口，stride 步长）
    Hn, Wn = ch.shape
    ny, nx = Hn // stride, Wn // stride
    out = np.zeros((ny, nx))
    for i in range(ny):
        for j in range(nx):
            y0, x0 = min(i * stride, Hn - k), min(j * stride, Wn - k)
            out[i, j] = ch[y0:y0 + k, x0:x0 + k].mean()
    return out

P128 = letterbox_params((180, 320), (128, 128))
print('letterbox:', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in P128.items()})
assert abs(P128['r'] - 0.4) < 1e-12 and P128['pad_y'] == 28.0 and P128['pad_x'] == 0.0
img0 = make_image(0)
lb0 = lb_image(img0, P128)
print('原图', img0.shape, ' letterbox 后', lb0.shape,
      ' 上下 padding 值 =', int(lb0[0, 0, 0]), '(应为 114)')
assert lb0[0, 0, 0] == 114 and lb0[64, 64, 0] != 114
print('✅ 迷你管线的图像部分就位。')

In [ ]:
STRIDE, KWIN, NET = 4, 8, 128
MEAN, STD = 0.45, 0.25
BOX_W = 12.0                      # 迷你模型预测的固定框宽（30px 原图 x r=0.4 = 12px）

def run_pipeline(img, p, cfg):
    # 返回 8 个阶段的张量。cfg 里的每个开关对应一种真实世界的 bug。
    st = {}
    st['1_img_u8'] = img
    st['2_lb_u8'] = lb_image(img, p)

    x = st['2_lb_u8'].astype(np.float32) / 255.0
    x = x if cfg.get('bgr_swap') else x[:, :, ::-1]        # 正确：BGR -> RGB
    x = ((x - MEAN) / STD).transpose(2, 0, 1).astype(np.float32)
    st['3_input_f32'] = x

    respR, respB = patch_mean(x[0], STRIDE, KWIN), patch_mean(x[2], STRIDE, KWIN)
    logits = np.stack([5.0 * respR - 4.0, 5.0 * respB - 4.0], axis=-1)     # (ny,nx,2)
    ny, nx, _ = logits.shape
    gy, gx = np.meshgrid(np.arange(ny), np.arange(nx), indexing='ij')
    cxs = (gx * STRIDE + KWIN / 2).astype(np.float64)
    cys = (gy * STRIDE + KWIN / 2).astype(np.float64)
    bx = np.stack([cxs - BOX_W / 2, cys - BOX_W / 2,
                   cxs + BOX_W / 2, cys + BOX_W / 2], axis=-1)             # (ny,nx,4)
    st['4_raw'] = np.concatenate([bx, logits], axis=-1).reshape(-1, 6).astype(np.float32)

    sc = sigmoid(st['4_raw'][:, 4:6].astype(np.float64))
    if cfg.get('double_sigmoid'):
        sc = sigmoid(sc)                                   # bug：模型内外各做了一次
    st['5_scores'] = sc.astype(np.float32)

    B = st['4_raw'][:, :4].astype(np.float64)
    bb = np.repeat(B, 2, axis=0)
    ss = sc.reshape(-1)
    cc = np.tile(np.array([0, 1]), len(B))
    m = ss >= cfg.get('score_thr', 0.35)
    bb, ss, cc = bb[m], ss[m], cc[m]
    if cfg.get('topk_before'):                             # bug：截断放在 NMS **之前**
        sel = score_order(ss)[:cfg['topk_before']]
        bb, ss, cc = bb[sel], ss[sel], cc[sel]
    ordr = score_order(ss)
    st['6_cands'] = np.concatenate([bb, ss[:, None], cc[:, None]], axis=1)[ordr]

    bb, ss, cc = st['6_cands'][:, :4], st['6_cands'][:, 4], st['6_cands'][:, 5]
    mode = cfg.get('nms_mode', 'classwise')
    io = cfg.get('iou_offset', 0)
    k = (nms_agnostic(bb, ss, cc, 0.5, io) if mode == 'agnostic'
         else nms_classwise(bb, ss, cc, 0.5, io))
    st['7_keep'] = st['6_cands'][k]

    out = st['7_keep'].copy()
    inv = inv_E1 if cfg.get('inverse_no_pad') else lb_inverse     # bug：逆变换漏减 pad
    out[:, :4] = np.stack([inv(row[:4], p) for row in st['7_keep']]) if len(out) else out[:, :4]
    st['8_final'] = out
    return st

REF = run_pipeline(img0, P128, {})
for k, v in REF.items():
    print(f'{k:<14s} shape={str(v.shape):<16s} dtype={v.dtype}')
print('\n最终检出（原图坐标 x1,y1,x2,y2,score,cls）:')
print(np.round(REF['8_final'], 1))
assert len(REF['7_keep']) >= 4, '4 块标志至少应检出 4 个框（紫色块占两类）'
gt_centers = np.array([[c[0], c[1]] for c in PLANTED], dtype=float)
det_centers = (REF['8_final'][:, :2] + REF['8_final'][:, 2:4]) / 2
for gc in gt_centers:
    assert np.min(np.linalg.norm(det_centers - gc, axis=1)) < 12.0, gc
print('\n✅ 参考管线跑通：每块标志都被检出，且还原到原图坐标后中心误差 < 12 px。')

In [ ]:
# ── 对拍器：逐阶段比对 + 自动定位第一个失配阶段 ──
TOL = [('1_img_u8',    'int',   1,     'JPEG 解码器实现 / BGR-RGB 顺序'),
       ('2_lb_u8',     'int',   2,     'resize 插值方式 / align_corners / pad 值与位置'),
       ('3_input_f32', 'float', 1e-5,  'mean-std 数值 / 除 255 时机 / 通道顺序 / NCHW 转置'),
       ('4_raw',       'float', 1e-3,  'engine 不同 / 输入不同（量化误差应在此容差内）'),
       ('5_scores',    'float', 1e-4,  'sigmoid 做了几次 / softmax-vs-sigmoid / 背景类偏移'),
       ('6_cands',     'set',   1e-4,  'score_thr 数值 / topK 位置与大小 / per-class vs global'),
       ('7_keep',      'set',   1e-4,  'IoU 的 +1 / iou_thr / class-wise-vs-agnostic / 排序稳定性'),
       ('8_final',     'set',   0.5,   'letterbox 逆变换 / clip 范围与顺序 / 坐标格式')]

def stage_diff(a, b, kind):
    if a.shape != b.shape:
        return float('inf'), f'shape {a.shape} vs {b.shape}'
    if a.size == 0:
        return 0.0, 'empty'
    d = float(np.abs(a.astype(np.float64) - b.astype(np.float64)).max())
    return d, f'max_abs_diff={d:.3e}'

def locate(ref, test, verbose=True):
    # 返回第一个超出容差的阶段名；全过则返回 None
    first = None
    for name, kind, tol, hint in TOL:
        d, msg = stage_diff(ref[name], test[name], kind)
        ok = d <= tol
        if verbose:
            flag = 'PASS' if ok else 'FAIL'
            print(f'  [{flag}] {name:<13s} tol={tol:<7g} {msg:<28s}' + ('' if ok else f'-> {hint}'))
        if not ok and first is None:
            first = name
            if verbose:
                continue
            break
    return first

print('自检：参考管线与自己对拍，应当全过')
assert locate(REF, run_pipeline(img0, P128, {}), verbose=False) is None
print('  [PASS] 全部 8 个阶段\n')

BUGS = [('通道顺序弄反（BGR/RGB）',   {'bgr_swap': True},        '3_input_f32'),
        ('sigmoid 做了两次',          {'double_sigmoid': True},  '5_scores'),
        ('topK 放在 NMS 之前',        {'topk_before': 6},        '6_cands'),
        ('NMS 用了 class-agnostic',   {'nms_mode': 'agnostic'},  '7_keep'),
        ('逆变换漏减 letterbox pad',  {'inverse_no_pad': True},  '8_final')]

for title, cfg, expect in BUGS:
    print(f'注入 bug：{title}')
    got = locate(REF, run_pipeline(img0, P128, cfg))
    print(f'  -> 定位到「{got}」，期望「{expect}」  {"✅" if got == expect else "❌"}\n')
    assert got == expect, (title, got, expect)

print('✅ 五种 bug，五次定位全部正确 —— 而且**不需要知道 bug 是什么**，')
print('   只需要按顺序走一遍 8 个阶段。八个阶段用二分只要 3 步。')
print('⚠️  最关键的一条纪律：第 4 阶段两侧必须跑**同一个 engine**（Python 也用 TRT Python API 加载），')
print('    否则量化误差混进来，你就分不清后面的分歧是模型造成的还是后处理造成的。')

## ✏️ 练习 1：IoU 约定何时会翻转抑制决策

实现 `will_flip(s, d, thr)`：两个边长 `s` 的正方形框沿对角线错开 `d` 像素，
判断 `offset=0` 与 `offset=1` 两种约定在阈值 `thr` 下是否给出**不同的抑制决策**。

- 抑制条件是 `IoU > thr`（等于阈值时保留）
- 返回 `True` 当且仅当两种约定一个抑制、一个保留

In [ ]:
def will_flip(s, d, thr):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert will_flip(8, 2, 0.41) is True,  '0.3913 <= 0.41 < 0.4336 -> 一边保留一边抑制'
assert will_flip(8, 2, 0.30) is False, '两种约定都 > 0.30 -> 都抑制'
assert will_flip(8, 2, 0.45) is False, '两种约定都 <= 0.45 -> 都保留'
assert will_flip(64, 16, 0.41) is False, '大框上 Δ 太小，翻不动'

flips = [s for s in range(4, 65) if will_flip(s, s / 4, 0.41)]
print('thr=0.41、相对位移 25% 时，会因为一个 +1 而翻转决策的框边长：', flips)
assert 8 in flips and 16 in flips
assert 32 not in flips and 64 not in flips
assert max(flips) < 25, '超过 ~20px 之后 Δ 就不足以跨过阈值了'
print(f'-> 临界边长约 {max(flips)} px。**TSR 的标志常年 10-30 px，正好全在危险区。**')
print('✅ 练习 1 通过：+1 约定是一个只在小目标上发作的 bug。')

## ✏️ 练习 2：逆变换 + clip + 退化过滤

实现 `inverse_and_clean(boxes_net, p, min_size=1.0)`，返回 `(boxes_orig, keep_mask)`：

1. 用**正确顺序**逆变换回原图坐标（先减 pad，再除 r）
2. clip 到 `[0, W] × [0, H]`（`W`/`H` 取自 `p`）
3. 宽或高 `<= min_size` 的框标记为丢弃（`keep_mask[i] = False`）

返回的 `boxes_orig` 是**全部**框（clip 后），`keep_mask` 标记哪些有效。

In [ ]:
def inverse_and_clean(boxes_net, p, min_size=1.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测（数字可口算：r=1/3, pad_y=140）——
tests = np.array([[100.0, 200.0, 140.0, 240.0],   # 正常框      -> [300, 180, 420, 300]
                  [300.0,  20.0, 340.0, 100.0],   # 全在上 pad  -> clip 后零高度 -> 丢弃
                  [ -5.0, 200.0,  20.0, 260.0],   # 左侧越界    -> x1 被 clip 到 0
                  [600.0, 200.0, 640.0, 260.0]])  # 右侧贴边    -> [1800, 180, 1920, 360]
out, mask = inverse_and_clean(tests, P640, min_size=1.0)
print('逆变换 + clip 结果：')
for i in range(len(tests)):
    print(f'  {tests[i]} -> {np.round(out[i],1)}   keep={bool(mask[i])}')

assert np.allclose(out[0], [300, 180, 420, 300])
assert np.allclose(out[1], [900, 0, 1020, 0])
assert np.allclose(out[2], [0, 180, 60, 360])
assert np.allclose(out[3], [1800, 180, 1920, 360])
assert mask.tolist() == [True, False, True, True]
assert out[:, 0].min() >= 0 and out[:, 2].max() <= P640['W']
assert out[:, 1].min() >= 0 and out[:, 3].max() <= P640['H']
print('\n✅ 练习 2 通过：padding 区里的假框被正确丢弃，越界框被 clip 到原图内。')
print('   注意第 2 行 —— 如果 clip 发生在**网络坐标**下，这个框会作为「贴在图像上边缘的检测」被上报。')

## ✏️ 练习 3：层次化 NMS

实现 `nms_hier(boxes, scores, cls, super_map, iou_thr=0.5, offset=0)`：

- `super_map` 是 `{类别 id: 超类名}`
- **同一超类内**做 class-agnostic NMS（细类互相抑制）
- **跨超类**互不抑制
- 返回升序排列的保留下标数组

两个退化情形必须自动成立：所有类同一超类 ⇒ 等价于 agnostic；每类各成一超类 ⇒ 等价于 class-wise。

In [ ]:
def nms_hier(boxes, scores, cls, super_map, iou_thr=0.5, offset=0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
kh = nms_hier(boxes_t, scores_t, cls_t, SUPER, 0.5)
print('层次化保留：', [CLS_NAME[int(cls_t[i])] for i in kh])
assert kh.tolist() == [0, 2, 3], kh.tolist()

all_same = {c: 'one' for c in range(4)}
assert np.array_equal(nms_hier(boxes_t, scores_t, cls_t, all_same, 0.5),
                      nms_agnostic(boxes_t, scores_t, cls_t, 0.5)), '退化情形 1'
each_own = {c: f's{c}' for c in range(4)}
assert np.array_equal(nms_hier(boxes_t, scores_t, cls_t, each_own, 0.5),
                      nms_classwise(boxes_t, scores_t, cls_t, 0.5)), '退化情形 2'

# 大场景上：层次化的保留数必须夹在 agnostic 与 class-wise 之间
b2, s2, _ = tsr_scene(30, np.random.default_rng(5), props=12)
c2 = np.random.default_rng(21).integers(0, 8, len(s2))
m2 = s2 >= 0.25
b2, s2, c2 = b2[m2], s2[m2], c2[m2]
SUP8 = {c: ('A' if c < 4 else 'B') for c in range(8)}      # 8 个细类归成 2 个超类
n_a = len(nms_agnostic(b2, s2, c2, 0.5))
n_h = len(nms_hier(b2, s2, c2, SUP8, 0.5))
n_c = len(nms_classwise(b2, s2, c2, 0.5))
print(f'\nagnostic {n_a} 个  <  层次化(2 超类) {n_h} 个  <  class-wise(8 类) {n_c} 个')
assert n_a < n_h < n_c
print('✅ 练习 3 通过：层次化是 agnostic 与 class-wise 之间的连续谱，由「什么能共存」这个物理先验决定。')

## ✏️ 练习 4：对拍定位器

实现 `first_mismatch(ref, test, spec)`：

- `spec` 是 `[(阶段名, 容差), ...]`，**按管线顺序排列**
- 逐阶段比较 `ref[name]` 与 `test[name]`：
  - shape 不同 → 直接判失配
  - 否则算 `max(|a - b|)`，`> 容差` 则判失配
- 返回**第一个**失配的阶段名；全部通过返回 `None`

In [ ]:
def first_mismatch(ref, test, spec):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
spec = [('a', 1.0), ('b', 1e-6), ('c', 0.5)]
ref_d = {'a': np.zeros(3), 'b': np.zeros((2, 2)), 'c': np.zeros(4)}
t1 = {'a': np.zeros(3), 'b': np.zeros((2, 2)), 'c': np.zeros(4)}
assert first_mismatch(ref_d, t1, spec) is None

t2 = dict(t1); t2['b'] = np.full((2, 2), 1e-3)
assert first_mismatch(ref_d, t2, spec) == 'b'

t3 = dict(t1); t3['b'] = np.zeros((3, 2))                  # shape 不同也算失配
assert first_mismatch(ref_d, t3, spec) == 'b'

t4 = dict(t1); t4['a'] = np.full(3, 0.5); t4['c'] = np.full(4, 9.0)
assert first_mismatch(ref_d, t4, spec) == 'c', 'a 在容差内，应跳过；第一个失配是 c'

# 接到真实管线上：五种 bug，五次定位
real_spec = [(name, tol) for name, kind, tol, hint in TOL]
for title, cfg, expect in BUGS:
    got = first_mismatch(REF, run_pipeline(img0, P128, cfg), real_spec)
    print(f'{title:<28s} -> {got}')
    assert got == expect, (title, got, expect)
assert first_mismatch(REF, run_pipeline(img0, P128, {}), real_spec) is None
print('\n✅ 练习 4 通过：不到 15 行代码，把「后处理为什么对不上」从猜谜变成了查表。')
print('   把它接进 CI，用 5-10 张精选回归图跑 —— 这是本模块唯一能**防止**问题的一条。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def will_flip(s, d, thr):
    A = [0, 0, s, s]
    B = [d, d, s + d, s + d]
    sup0 = iou_pair(A, B, 0) > thr          # 抑制条件：IoU > thr
    sup1 = iou_pair(A, B, 1) > thr
    return bool(sup0 != sup1)

In [ ]:
# 练习 2 参考答案
def inverse_and_clean(boxes_net, p, min_size=1.0):
    b = np.asarray(boxes_net, dtype=float).copy()
    b[:, [0, 2]] = (b[:, [0, 2]] - p['pad_x']) / p['r']      # 先减 pad，再除 r
    b[:, [1, 3]] = (b[:, [1, 3]] - p['pad_y']) / p['r']
    b[:, [0, 2]] = np.clip(b[:, [0, 2]], 0, p['W'])          # clip 在**原图**坐标下
    b[:, [1, 3]] = np.clip(b[:, [1, 3]], 0, p['H'])
    keep = (b[:, 2] - b[:, 0] > min_size) & (b[:, 3] - b[:, 1] > min_size)
    return b, keep

In [ ]:
# 练习 3 参考答案
def nms_hier(boxes, scores, cls, super_map, iou_thr=0.5, offset=0):
    sup = np.array([super_map[int(c)] for c in cls])
    keep = []
    for u in np.unique(sup):                                 # 跨超类互不抑制
        idx = np.where(sup == u)[0]
        keep.append(idx[nms(boxes[idx], scores[idx], iou_thr, offset)])   # 超类内 agnostic
    return np.sort(np.concatenate(keep)) if keep else np.zeros(0, dtype=int)

In [ ]:
# 练习 4 参考答案
def first_mismatch(ref, test, spec):
    for name, tol in spec:
        a, b = np.asarray(ref[name]), np.asarray(test[name])
        if a.shape != b.shape:
            return name
        if a.size and float(np.abs(a.astype(np.float64) - b.astype(np.float64)).max()) > tol:
            return name
    return None

---
## 🧪 真实工程胶囊：后处理对齐 spec + C++ 管线检查单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# 第一部分：后处理 SPEC（这份文件是三方实现的唯一真相源）
#   把它放进仓库，Python 参考实现 / C++ 实现 / TRT plugin 配置都必须对齐到它
# ══════════════════════════════════════════════════════════════════════
postprocess_spec:
  activation:      sigmoid          # 在**模型外**做；ONNX 里禁止出现 Sigmoid 节点
  background_class: none            # sigmoid 头无背景类，class_id 从 0 开始
  box_format:      cxcywh           # 模型原始输出格式
  box_normalized:  false            # 像素坐标，**相对于网络输入 640x640（含 padding）**
  score_threshold: 0.05             # 只用于控计算量；真正的工作点阈值在 spec 之外二次过滤
  topk_before_nms: 4096             # ← 按「单帧最大过阈候选数」留 2-3 倍余量
  nms:
    iou_threshold: 0.55
    iou_offset:    0                # ← **COCO 连续坐标语义，不加 +1**
    mode:          hierarchical     # 超类内 agnostic，跨超类不抑制
    superclass_map: superclass_v3.yaml
    suppress_cond: "iou > thr"      # 等于阈值时**保留**
    sort_key:      "(-score, flat_index)"   # ← 确定性全序，INT8 下并列分数不再有歧义
  max_det:         300              # ← NMS **之后**的截断，与 topk_before_nms 是两回事
  inverse:
    order:         "subtract_pad_then_divide_r"     # ← 顺序写死在 spec 里
    clip_space:    original_image                   # 不是网络输入！
    min_box_size:  1.0                              # 退化框直接丢弃
  emit_competing_hypotheses: true   # 被同超类抑制掉的 (类别, 分数) 随框上报给下游

# ══════════════════════════════════════════════════════════════════════
# 第二部分：SPEC 测试用例（每一条对应本模块讲过的一个坑）
# ══════════════════════════════════════════════════════════════════════
# T1  IoU 恰好等于阈值的一对框            -> 验证 ">" 而不是 ">="
# T2  分数完全相同的一对框                -> 验证确定性排序（INT8 必测）
# T3  面积为 0 / 坐标为负 / 超出边界的框   -> 验证退化过滤
# T4  单帧候选数 > topk_before_nms 的密集帧 -> 验证截断位置（NMS 前 vs 后）
# T5  同位置双类高分（限速60/80）          -> 验证层次化 NMS 与竞争假设上报
# T6  主牌 + 紧贴的辅助牌                  -> 验证跨超类不抑制
# T7  完全落在 letterbox padding 区的候选  -> 验证 clip 在原图坐标下做
# T8  非对称长宽比输入（如 1920x1084）     -> 验证亚像素取整一致
#
# ══════════════════════════════════════════════════════════════════════
# 第三部分：端到端对拍工具（项目第一天就要有，不是出事之后）
# ══════════════════════════════════════════════════════════════════════
# C++ 侧编译 debug 模式：./tsr_infer --image a.jpg --dump-stage=all --dump-dir=/tmp/cpp
#   写 .npy 不需要任何依赖，格式很简单（魔数 NUMPY + header + 裸数据），约 100 行
# Python 侧：python tools/xdiff.py --ref /tmp/py --test /tmp/cpp --spec stages.yaml
#
# **纪律 1**：两侧第 4 阶段必须跑同一个 engine（Python 用 tensorrt 的 Python API 加载）
#             否则量化误差混进来，模型层与后处理层就分不开了
# **纪律 2**：第 6/7 阶段的容差是「个数完全相等」，不是「数值接近」
#             因为 NMS 是不连续算子，1 个候选的差别会放大成几十个框的差别
# **纪律 3**：回归图要精选 —— 稀疏帧 / 密集帧 / INT8 并列帧 / 贴边缘帧 / 极端长宽比帧
#
# ══════════════════════════════════════════════════════════════════════
# 第四部分：C++ 推理管线检查单
# ══════════════════════════════════════════════════════════════════════
# [内存]
#   □ 稳态运行期**零** cudaMalloc / cudaFree（它们会隐式同步整个 device）
#   □ 动态 shape 下按 max profile 一次性分配
#   □ host 侧全部用 cudaHostAlloc（pinned）—— pageable 上的 MemcpyAsync 实际是同步的
#   □ 输入/输出 buffer 双缓冲，避免第 k 帧后处理与第 k+1 帧推理竞争
# [流与同步]
#   □ 预处理 kernel / enqueueV3 / 后处理 kernel / D2H 全部挂同一条 stream
#   □ **每帧只允许一次 cudaStreamSynchronize**（在读取最终结果时）
#   □ 全局搜一遍 cudaDeviceSynchronize / 非 Async 的 Memcpy / Memset，确认没有漏网的
#   □ 用 Nsight Systems 看 CPU 与 GPU 泳道是否重叠；交替 = 有多余同步
# [零拷贝]
#   □ 判据是「这块数据被读几次」：只读一次 -> 零拷贝划算；反复读 -> 拷到 device memory
#   □ mapped pinned memory 在 Orin 上通常 uncached，**CPU 读它非常慢**
# [线程模型]
#   □ 车端选延迟不选吞吐：单线程 + GPU 预处理 或 双缓冲 2 线程
#   □ 3 阶段流水线吞吐最高但延迟约等于三段之和 —— 那是数据中心的最佳实践，不是车端的
# [相机对接]
#   □ 时间戳用**曝光中点**，不是帧到达时刻（100 km/h 下 10 ms = 0.28 m 纵向误差）
#   □ 队列满时丢**最老**的帧，且丢帧必须计数并把 frame_id 传给下游
#   □ 多相机硬件同步，否则跨相机关联会失败
# [错误处理]
#   □ CUDA 错误是粘性的：每次调用都 check，一次 error 之后所有调用返回同一个 error
#   □ 对拍用容差不用 ==：TRT 部分 kernel 用 atomic 累加，浮点加法不满足结合律
'''
print(RECIPE)
for token in ['iou_offset', 'topk_before_nms', 'max_det', 'sort_key',
              'subtract_pad_then_divide_r', 'clip_space', 'cudaHostAlloc',
              '曝光中点', '同一个 engine']:
    assert token in RECIPE, token
print('✅ 胶囊覆盖：SPEC / 测试用例 / 对拍工具纪律 / C++ 管线（内存·流·零拷贝·线程·相机·错误）')

### 小结

- **后处理是唯一一段「两份实现、零份约束」的代码。** 预处理和模型至少有同一份 ONNX 作锚点，
  后处理只有口头约定。它的 bug 不报错、不崩溃，只把 mAP 从 0.82 磨到 0.71，
  **然后你会花两周去怀疑量化。**
- **IoU 的 `+1` 是尺度相关的**：$\Delta(s) \sim \mathcal{O}(1/s)$，8 px 框差 0.042，64 px 框只差 0.006。
  `thr=0.41`、相对位移 25% 时，**临界边长约 18 px** —— TSR 的标志正好全在危险区。
  一致性是「标注 / 训练+评测 / 部署」**三方**一致，不是两方。
- **NMS 前的 topK ≠ NMS 后的 max_det。** 前者作用在未去重的候选上：密集帧里 topK=100 只覆盖了
  26/60 个目标，而不截断能覆盖 54/60；**稀疏帧上两者完全一样**——所以离线评测测不出来。
- **`σ(σ(x))` 是最阴险的 bug**：分数全被压进 (0.5, 0.731)，阈值彻底失效、候选数暴涨、
  线上 FP 爆炸，而 **AP 一个字节都不变**（单调变换不改排序）。诊断只要 `print(scores.min())`。
- **4 px 的系统性偏移**：100 px 大牌 IoU 还有 **0.855**（永远发现不了），
  12 px 小牌只剩 **0.286**（IoU=0.5 口径下算完全漏检）。
  **凡是「只有小目标桶掉点」，第一怀疑对象永远是坐标变换，不是模型能力。**
- **NMS 模式在编码「什么在物理世界里能共存」**：同一块牌子不能既是限速 60 又是限速 80（该抑制），
  主牌和辅助牌是两块物理牌子（不该抑制）→ **层次化 NMS**。
  而且被抑制的类别与分数应作为「竞争假设」上报，让下游做多帧贝叶斯累积而不是对硬判决投票。
- **防线不是「小心」，是「spec + 逐阶段对拍 + CI」**：8 个阶段、逐阶段容差、二分定位；
  五种注入 bug 全部一次命中。关键纪律是**两侧第 4 阶段必须跑同一个 engine**，
  以及**在不连续算子处（阈值、排序、NMS）把容差收紧成「完全相等」**。

下一站：**模块 05 · 性能剖析与延迟工程** —— 语义都对齐了，接下来是它跑得够不够快、够不够稳。